structured output

using pydentic

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, client=<groq.resources.chat.completions.Completions object at 0x000001EC06B4F0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EC06B4FB60>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the Movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The Director of the movie")
    rating:float=Field(description="The movies ratings out of 10")    

In [9]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, client=<groq.resources.chat.completions.Completions object at 0x000001EC06B4F0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EC06B4FB60>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the Movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The Director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies ratings out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'M

In [14]:
response=model.invoke("provide details about the movie batman")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request:** The user asked for "details about the movie batman". This is quite broad since there are multiple Batman movies. I need to clarify which one they mean, but also provide a comprehensive overview that covers the major ones, or ask for clarification while giving a structured response.\n\n2.  **Identify Key Ambiguity:** "Batman" has been adapted into numerous films across different eras and franchises:\n   - Tim Burton\'s Batman (1989)\n   - Batman Returns (1992)\n   - Batman Forever (1995)\n   - Batman & Robin (1997)\n   - Christopher Nolan\'s The Dark Knight Trilogy (2005-2012)\n   - DC Extended Universe (DCEU) films: Batman v Superman (2016), Suicide Squad (cameo), Justice League (2017)\n   - Matt Reeves\' The Batman (2022)\n   - Animated films (many)\n   - Upcoming: The Batman Part II (2026)\n\n3.  **Determine Response Strategy:** \n   - Acknowledge the ambiguity\n   - Provide a structured ove

In [13]:
response1=model_with_structure.invoke("provide details about the movie batman begins")
response1

Movie(title='Batman Begins', year=2005, director='Christopher Nolan', rating=8.2)

Message output parsed structure

In [15]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the Movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The Director of the movie")
    rating:float=Field(...,description="The movies ratings out of 10")  

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response1=model_with_structure.invoke("provide details about the movie batman begins")
response1  

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "provide details about the movie batman begins"\n   - Key entity: "Batman Begins" (movie)\n   - Expected details: title, year, director, rating (based on the available function schema)\n\n2.  **Identify Available Function:**\n   - Function: `Movie`\n   - Parameters: `title` (string, required), `year` (integer, required), `director` (string, required), `rating` (number, required)\n\n3.  **Determine Required Information for Function Call:**\n   - I need to provide all four required parameters: title, year, director, rating.\n   - The user only provided the title ("batman begins").\n   - I need to fill in the missing information based on my knowledge:\n     - Title: "Batman Begins"\n     - Year: 2005\n     - Director: Christopher Nolan\n     - Rating: I need to provide a rating out of 10. Common ratings for Batman Begins: IMDb ~8.2, Rotten Toma

nested structure

In [ ]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None,description="Budget is millions USD")

model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("provide me the detail of movie Batman Begins")
response
    

AttributeError: 'ChatGroq' object has no attribute 'with_structure'